# Pattern-Matching NILM

## Imports

In [16]:
import os
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from fastdtw import fastdtw

## Directories

In [17]:
base_dir = os.getcwd()
processed_data_dir = os.path.join(base_dir, 'processed_data')

results_dir = os.path.join(base_dir, 'results')
os.makedirs(results_dir, exist_ok=True)

## Load Data

In [18]:
processed_data_filepath = os.path.join(processed_data_dir, 'ampds_dataset.npz')
data = np.load(processed_data_filepath)

# Check Data
print(data.files)

# check shape of x_norm and y_norm
print("x_norm shape:", data["x_norm"].shape)
print("y_norm shape:", data["y_norm"].shape)

# Check the index shapes
# Create a table of input/output index shapes for each dataset split
dataset_names = sorted({
    key.rsplit('_', 1)[0]
    for key in data.files
    if key.endswith(('_inp', '_out'))})

index_shapes = pd.DataFrame([
    {
        'Dataset': name,
        'Input index shape': data[f'{name}_inp'].shape if f'{name}_inp' in data.files else None,
        'Output index shape': data[f'{name}_out'].shape if f'{name}_out' in data.files else None}
    for name in dataset_names])

display(index_shapes)

['x_norm', 'y_norm', 'x_min', 'x_max', 'y_min', 'y_max', 'train_inp', 'train_out', 'val_inp', 'val_out', 'test_inp', 'test_out', 'T_limit', 'train_val_test_split', 'window_length', 'stride']
x_norm shape: (1000, 6)
y_norm shape: (1000, 22, 4)


,Dataset,Input index shape,Output index shape
0,test,"(83,)","(83,)"
1,train,"(492,)","(492,)"
2,val,"(164,)","(164,)"


## Create Template Database

Pattern-matching algorithms work by comparing new inputs with examples stored from the training data. Here, the normalizes data and training input/output indices already define those examples, so there's no need to build a separate template database.

## Sample Generation

To create a sample, we use the normalized aggregate and appliance profiles (`y_norm` and `x_norm`), along with the input/output indices and window length, to select an input window and its corresponding target.

In [19]:
def generate_sample(x, y, i_inp, i_out, window_length):
    x_win = x[i_inp : i_inp + window_length]
    y_target = y[i_out]
    return x_win, y_target

## Time-series data frame comparison

### Feature distance

One way to compare time-series frames is to summarize each with a feature vector, then measure the difference between those vectors. Possible features include the mean, minimum, maximum, standard deviation, root-mean-square (RMS), and energy.

In [20]:
def compute_feature_vector(x_win, feature_selection='all'):

    funcs = {
        'mean': lambda x: np.mean(x),
        'min': lambda x: np.min(x),
        'max': lambda x: np.max(x),
        'std': lambda x: np.std(x),
        'rms': lambda x: np.mean(x**2),
        'energy': lambda x: np.sum(x**2)}

    if feature_selection == 'all': feature_selection = list(funcs.keys())
    out = np.column_stack([funcs[name](x_win) for name in feature_selection])
    return out

In [21]:
def feature_distance(vector1, vector2): return np.linalg.norm(vector1 - vector2)

Recomputing template features for every test-frame comparison would be inefficient. Instead, we compute and store each template's feature vector once, then use a lookup keyed by the template's input index to retrieve it.

In [22]:
feature_lookup = {
    int(i): compute_feature_vector(data['x_norm'][i : i + data['window_length'].item()])
    for i in data['train_inp']}

In [23]:
print("Feature lookup dictionary length:", len(feature_lookup))

columns = ["Mean", "Min", "Max", "Std", "RMS", "Energy"]
df = pd.DataFrame([v.ravel() for v in list(feature_lookup.values())[:5]],
                  index=list(feature_lookup)[:5], columns=columns)
df.index.name = "Template Start Index"
display(df.reset_index().style.hide(axis="index"))

Feature lookup dictionary length: 492


Template Start Index,Mean,Min,Max,Std,RMS,Energy
291,0.064807,0.000000,0.320320,0.108713,0.016018,2.883313
256,0.227654,0.000000,0.394043,0.122000,0.066710,12.007883
855,0.222738,0.000000,0.884885,0.302812,0.141307,25.435296
228,0.075722,0.000000,0.390059,0.102117,0.016162,2.909131
53,0.109409,0.000000,1.000000,0.160628,0.037772,6.798902


### Correlation

Correlation provides another way to comprare two time-series frames. This function calculates their Pearson Correlation. Pearson correlation measures how strongly two variables move together in a linear way. The function flattens each frame into a one-dimensional array, computes their correlation matrix, and selects the value comparing the two frames. The result ranges from -1 (opposing patterns) to +1 (strongly similar patterns). A value near 0 indicates little linear relationship.

In [24]:
def compute_correlation(frame1, frame2): return np.corrcoef(frame1.flatten(), frame2.flatten())[0, 1]

### Dynamic Time Warping

Dynamic time warping (DTW) compares two time-series segments by aligning their points flexibly, allowing similar patterns to occur at different speeds or be shifted in time. It finds the lowest-cost alignment between the sequences. The `distance` is the accumulates cost of the best alisngment. A smaller DTW distance means they are more similar. The `path` is the alignment as pairs of indices. Each pair indicates which point in the first sequence corresponds to a point in the second. A point may appear in multiple pairs to account for timing differences.

In [25]:
def compute_dtw(frame1, frame2):
    
    distance, path = fastdtw(frame1.flatten(), frame2.flatten())
    return distance, path

## Pattern Matching Inference

In [52]:
def predict(data, methods='all', feature_selection='all', feature_lookup=None):
    
    columns = [
        'test frame start index',
        'template start index',
        'feature distance',
        'pearson correlation',
        'dtw distance']

    if methods == 'all': methods = ['feature_distance', 'pearson_correlation', 'dtw_distance']

    window_length = int(data['window_length'].item())
    train_indices = data['train_inp']
    rows = []

    for test_index in tqdm(data['test_inp'], desc='Predicting'):
        test_index = int(test_index)
        test_frame = data['x_norm'][test_index:test_index + window_length]
        test_features = (
            compute_feature_vector(test_frame, feature_selection)
            if 'feature_distance' in methods else None)

        for template_index in train_indices:

            template_index = int(template_index)
            row = {
                'test frame start index': test_index,
                'template start index': template_index,
                'feature distance': np.nan,
                'pearson correlation': np.nan,
                'dtw distance': np.nan}

            if 'feature_distance' in methods: 
                template_features = feature_lookup[template_index]
                row['feature distance'] = feature_distance(test_features, template_features)

            if 'pearson_correlation' in methods:
                template_frame = data['x_norm'][template_index:template_index + window_length]
                row['pearson correlation'] = compute_correlation(test_frame, template_frame)

            if 'dtw_distance' in methods:
                template_frame = data['x_norm'][template_index:template_index + window_length]
                row['dtw distance'] = compute_dtw(test_frame, template_frame)[0]

            rows.append(row)

    return pd.DataFrame(rows, columns=columns)


In [53]:
results = predict(data, methods=['feature_distance'], feature_selection='all', feature_lookup=feature_lookup)
display(results.head(10).style.hide(axis="index"))

Predicting:   0%|          | 0/83 [00:00<?, ?it/s]

test frame start index,template start index,feature distance,pearson correlation,dtw distance
935,291,27.424110,nan,nan
935,256,18.300391,nan,nan
935,855,4.863509,nan,nan
935,228,27.396730,nan,nan
935,53,23.500427,nan,nan
935,373,21.739752,nan,nan
935,832,0.903420,nan,nan
935,288,27.482072,nan,nan
935,701,14.437018,nan,nan
935,722,0.174199,nan,nan
